# NVS Benchmark - Notebook de Execucao

Notebook principal para checks, metricas e relatorio. Funciona em ambiente local e no Colab.

In [ ]:
# Celula 1: bootstrap (local/colab)
from pathlib import Path
import os
import subprocess
import sys

def detect_colab() -> bool:
    try:
        __import__("google.colab")
        return True
    except Exception:
        return False

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    return start

IS_COLAB = detect_colab()
if IS_COLAB and Path("/content/TCC").exists():
    project_root = Path("/content/TCC")
else:
    project_root = find_project_root(Path.cwd())

os.chdir(project_root)
print("Ambiente Colab:", IS_COLAB)
print("Raiz do projeto:", project_root)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

def run_cli(*args):
    cmd = [sys.executable, "-m", "nvs_benchmark.cli", *args]
    print("Executando:", ' '.join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# Celula 2: pipeline de benchmark
snapshot_file = "./artifacts/metrics/latest_preview.json"

run_cli("status")
run_cli("methods-check", "--output-dir", "./artifacts", "--log-dir", "./logs")
run_cli("metrics-check",
        "--output-dir", "./artifacts",
        "--snapshot-file", snapshot_file,
        "--log-dir", "./logs")
run_cli("report-generate",
        "--snapshot-file", snapshot_file,
        "--output-dir", "./artifacts/reports",
        "--report-name", "benchmark_report",
        "--log-dir", "./logs")

print("Snapshot:", snapshot_file)
print("Relatorio:", "./artifacts/reports/benchmark_report.html")

In [ ]:
# Celula 3: preview
if IS_COLAB:
    print("No Colab, prefira abrir o relatorio HTML gerado em artifacts/reports.")
    display_mod = __import__("IPython.display", fromlist=["IFrame"])
    IFrame = getattr(display_mod, "IFrame")
    IFrame(src="./artifacts/reports/benchmark_report.html", width=1200, height=700)
else:
    ui_cmd = [
        sys.executable, "-m", "nvs_benchmark.cli", "ui-preview",
        "--host", "127.0.0.1",
        "--port", "8765",
        "--metrics-file", "./artifacts/metrics/latest_preview.json",
        "--scene-transforms-file", "./data/_smoke/blender/transforms_train.json",
    ]
    print("Iniciando UI:", ' '.join(ui_cmd))
    subprocess.Popen(ui_cmd)